### Setup and data audit

In [1]:
import numpy as np
import pandas as pd

from alpha_research.config.paths import PROCESSED_DATA_DIR
from alpha_research.signal_processing import (
    add_sector_neutral_factor,
    process_factor_columns,
)
from alpha_research.validation import calculate_ic_by_horizon

factor_panel = pd.read_parquet(
    PROCESSED_DATA_DIR / "factor_panel.parquet"
)

required_columns = [
    "date",
    "ticker",
    "sector",
    "ret_1d",
    "mom_12_1m_raw",
    "mom_12_1m_z",
    "mom_12_1m_sector_neutral_z",
    "realised_vol_63_raw",
    "idio_vol_63_raw",
    "forward_ret_1d",
    "forward_ret_5d",
]

missing_columns = [
    column
    for column in required_columns
    if column not in factor_panel.columns
]

assert not missing_columns, f"Missing columns: {missing_columns}"

momentum_panel = (
    factor_panel
    .sort_values(["ticker", "date"])
    .reset_index(drop=True)
    .copy()
)

momentum_panel[required_columns].info()

<class 'pandas.DataFrame'>
RangeIndex: 284249 entries, 0 to 284248
Data columns (total 11 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   date                        284249 non-null  datetime64[ms]
 1   ticker                      284249 non-null  str           
 2   sector                      284249 non-null  str           
 3   ret_1d                      284148 non-null  float64       
 4   mom_12_1m_raw               259036 non-null  float64       
 5   mom_12_1m_z                 259036 non-null  float64       
 6   mom_12_1m_sector_neutral_z  251119 non-null  float64       
 7   realised_vol_63_raw         277936 non-null  float64       
 8   idio_vol_63_raw             277936 non-null  float64       
 9   forward_ret_1d              284148 non-null  float64       
 10  forward_ret_5d              283744 non-null  float64       
dtypes: datetime64[ms](1), float64(8), str(2)
memory us

### Construct candidate signals

In [2]:
formation_window = 252 - 21
minimum_formation_observations = 189

momentum_panel["formation_vol_12_1m_raw"] = (
    momentum_panel
    .groupby("ticker")["ret_1d"]
    .transform(
        lambda series: (
            series
            .shift(21)
            .rolling(
                window=formation_window,
                min_periods=minimum_formation_observations,
            )
            .std()
        )
    )
    * np.sqrt(252)
)

valid_formation_vol = (
    momentum_panel["formation_vol_12_1m_raw"]
    .where(momentum_panel["formation_vol_12_1m_raw"] > 0)
)

valid_realised_vol = (
    momentum_panel["realised_vol_63_raw"]
    .where(momentum_panel["realised_vol_63_raw"] > 0)
)

valid_idio_vol = (
    momentum_panel["idio_vol_63_raw"]
    .where(momentum_panel["idio_vol_63_raw"] > 0)
)

# Primary specification: momentum and risk use the same formation period.
momentum_panel["risk_adjusted_mom_aligned_raw"] = (
    momentum_panel["mom_12_1m_raw"]
    / valid_formation_vol
)

# Diagnostic: scale momentum by recent total volatility.
momentum_panel["risk_adjusted_mom_recent_vol_raw"] = (
    momentum_panel["mom_12_1m_raw"]
    / valid_realised_vol
)

# Diagnostic: scale momentum by recent market-model residual risk.
momentum_panel["risk_adjusted_mom_idio_vol_raw"] = (
    momentum_panel["mom_12_1m_raw"]
    / valid_idio_vol
)

### Distribution and coverage

In [3]:
raw_risk_adjusted_columns = [
    "formation_vol_12_1m_raw",
    "risk_adjusted_mom_aligned_raw",
    "risk_adjusted_mom_recent_vol_raw",
    "risk_adjusted_mom_idio_vol_raw",
]

risk_adjusted_distribution_summary = (
    momentum_panel[raw_risk_adjusted_columns]
    .agg(
        [
            "count",
            "mean",
            "std",
            "min",
            "median",
            "max",
            "skew",
        ]
    )
    .T
)

risk_adjusted_distribution_summary["coverage"] = (
    momentum_panel[raw_risk_adjusted_columns]
    .notna()
    .mean()
)

risk_adjusted_distribution_summary

,count,mean,std,min,median,max,skew,coverage
formation_vol_12_1m_raw,263236.0,0.285111,0.119422,0.084557,0.254735,0.961538,1.549231,0.926075
risk_adjusted_mom_aligned_raw,259036.0,0.664392,1.045587,-2.123875,0.528721,12.892534,0.969389,0.911300
risk_adjusted_mom_recent_vol_raw,259036.0,0.736572,1.133255,-1.937496,0.564984,13.499275,1.148106,0.911300
risk_adjusted_mom_idio_vol_raw,259036.0,0.890392,1.395972,-3.205978,0.690319,16.865016,1.058519,0.911300


### Process and sector-neutralise

In [4]:
risk_adjusted_factor_map = {
    "risk_adjusted_mom_aligned_raw": (
        "risk_adjusted_mom_aligned"
    ),
    "risk_adjusted_mom_recent_vol_raw": (
        "risk_adjusted_mom_recent_vol"
    ),
    "risk_adjusted_mom_idio_vol_raw": (
        "risk_adjusted_mom_idio_vol"
    ),
}

momentum_panel = process_factor_columns(
    momentum_panel,
    factor_map=risk_adjusted_factor_map,
    lower_quantile=0.01,
    upper_quantile=0.99,
)

for factor_prefix in risk_adjusted_factor_map.values():
    momentum_panel = add_sector_neutral_factor(
        momentum_panel,
        factor_column=f"{factor_prefix}_winsorised",
        output_column=f"{factor_prefix}_sector_neutral_z",
        sector_column="sector",
        min_sector_observations=3,
    )

### Initial IC comparison

In [5]:
risk_adjusted_signals = {
    "12–1 momentum — raw": "mom_12_1m_z",
    "12–1 momentum — sector neutral": (
        "mom_12_1m_sector_neutral_z"
    ),
    "Risk-adjusted momentum — aligned": (
        "risk_adjusted_mom_aligned_z"
    ),
    "Risk-adjusted momentum — aligned, sector neutral": (
        "risk_adjusted_mom_aligned_sector_neutral_z"
    ),
    "Risk-adjusted momentum — recent realised vol": (
        "risk_adjusted_mom_recent_vol_z"
    ),
    "Risk-adjusted momentum — recent realised vol, sector neutral": (
        "risk_adjusted_mom_recent_vol_sector_neutral_z"
    ),
    "Risk-adjusted momentum — idio vol": (
        "risk_adjusted_mom_idio_vol_z"
    ),
    "Risk-adjusted momentum — idio vol, sector neutral": (
        "risk_adjusted_mom_idio_vol_sector_neutral_z"
    ),
}

risk_adjusted_ic_results = []

for signal_name, factor_column in risk_adjusted_signals.items():
    result = calculate_ic_by_horizon(
        panel=momentum_panel,
        factor_column=factor_column,
        forward_return_columns=[
            "forward_ret_1d",
            "forward_ret_5d",
        ],
        method="spearman",
        min_observations=30,
    )

    result["signal"] = signal_name
    risk_adjusted_ic_results.append(result)

risk_adjusted_ic_summary = pd.concat(
    risk_adjusted_ic_results,
    ignore_index=True,
)

risk_adjusted_ic_summary[
    [
        "signal",
        "forward_return_column",
        "count",
        "mean_ic",
        "std_ic",
        "ic_ir",
        "t_stat",
        "positive_fraction",
    ]
]

,signal,forward_return_column,count,mean_ic,std_ic,ic_ir,t_stat,positive_fraction
0,12–1 momentum — raw,forward_ret_1d,2638.0,0.019428,0.283058,0.068636,3.525242,0.541698
1,12–1 momentum — raw,forward_ret_5d,2634.0,0.019718,0.277559,0.071042,3.646044,0.555809
2,12–1 momentum — sector neutral,forward_ret_1d,2638.0,0.015692,0.192945,0.081330,4.177206,0.539424
3,12–1 momentum — sector neutral,forward_ret_5d,2634.0,0.018429,0.185880,0.099145,5.088367,0.553531
4,Risk-adjusted momentum — aligned,forward_ret_1d,2638.0,0.018323,0.270220,0.067808,3.482724,0.542456
5,Risk-adjusted momentum — aligned,forward_ret_5d,2634.0,0.016039,0.265135,0.060494,3.104716,0.547077
6,"Risk-adjusted momentum — aligned, sector neutral",forward_ret_1d,2638.0,0.015688,0.189871,0.082623,4.243635,0.541698
7,"Risk-adjusted momentum — aligned, sector neutral",forward_ret_5d,2634.0,0.015826,0.183723,0.086142,4.421036,0.541762
8,Risk-adjusted momentum — recent realised vol,forward_ret_1d,2638.0,0.019149,0.267801,0.071504,3.672563,0.546247
9,Risk-adjusted momentum — recent realised vol,forward_ret_5d,2634.0,0.017198,0.262173,0.065599,3.366706,0.548216


## Diagnostics on idio vol adjusted momentum

### Daily rank-correlation diagnostic
Does risk-adjusted momentum differ substantially from raw momentum and idio vol?

In [6]:
diagnostic_columns = {
    "Risk-adjusted momentum vs momentum": (
        "risk_adjusted_mom_idio_vol_z",
        "mom_12_1m_z",
    ),
    "Risk-adjusted momentum vs idio vol": (
        "risk_adjusted_mom_idio_vol_z",
        "idio_vol_63_raw",
    ),
}

correlation_rows = []

for date, group in momentum_panel.groupby("date"):
    for pair_name, (left_column, right_column) in (
        diagnostic_columns.items()
    ):
        valid = group[
            [left_column, right_column]
        ].dropna()

        if len(valid) < 30:
            continue

        correlation_rows.append(
            {
                "date": date,
                "pair": pair_name,
                "correlation": valid[left_column].corr(
                    valid[right_column],
                    method="spearman",
                ),
            }
        )

daily_risk_adjusted_correlations = pd.DataFrame(
    correlation_rows
)

risk_adjusted_correlation_summary = (
    daily_risk_adjusted_correlations
    .groupby("pair")["correlation"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        std="std",
        min="min",
        max="max",
    )
)

risk_adjusted_correlation_summary

,count,mean,median,std,min,max
pair,,,,,,
Risk-adjusted momentum vs idio vol,2639,-0.176865,-0.181346,0.157551,-0.547812,0.402342
Risk-adjusted momentum vs momentum,2639,0.954772,0.964799,0.028012,0.834341,0.990847


### Residualise against ordinary momentum

In [7]:
def cross_sectional_rank_residual(
    group,
    target_column,
    explanatory_column,
):
    result = pd.Series(
        np.nan,
        index=group.index,
        dtype=float,
    )

    valid = group[
        [target_column, explanatory_column]
    ].dropna()

    if len(valid) < 30:
        return result

    target_rank = valid[target_column].rank(
        method="average",
        pct=True,
    )
    explanatory_rank = valid[explanatory_column].rank(
        method="average",
        pct=True,
    )

    target_standardised = (
        target_rank - target_rank.mean()
    ) / target_rank.std(ddof=0)

    explanatory_standardised = (
        explanatory_rank - explanatory_rank.mean()
    ) / explanatory_rank.std(ddof=0)

    design_matrix = np.column_stack(
        [
            np.ones(len(valid)),
            explanatory_standardised.to_numpy(),
        ]
    )

    coefficients = np.linalg.lstsq(
        design_matrix,
        target_standardised.to_numpy(),
        rcond=None,
    )[0]

    residuals = (
        target_standardised.to_numpy()
        - design_matrix @ coefficients
    )

    result.loc[valid.index] = residuals
    return result

In [8]:
momentum_panel[
    "risk_adjusted_mom_idio_incremental"
] = (
    momentum_panel
    .groupby("date", group_keys=False)
    .apply(
        lambda group: cross_sectional_rank_residual(
            group=group,
            target_column="risk_adjusted_mom_idio_vol_z",
            explanatory_column="mom_12_1m_z",
        ),
    )
)

momentum_panel[
    "risk_adjusted_mom_idio_sector_incremental"
] = (
    momentum_panel
    .groupby("date", group_keys=False)
    .apply(
        lambda group: cross_sectional_rank_residual(
            group=group,
            target_column=(
                "risk_adjusted_mom_idio_vol_sector_neutral_z"
            ),
            explanatory_column=(
                "mom_12_1m_sector_neutral_z"
            ),
        ),
    )
)

In [ ]:
# Check that residual correlation -> 0

residual_correlation_rows = []

residual_pairs = {
    "Raw residual": (
        "risk_adjusted_mom_idio_incremental",
        "mom_12_1m_z",
    ),
    "Sector-neutral residual": (
        "risk_adjusted_mom_idio_sector_incremental",
        "mom_12_1m_sector_neutral_z",
    ),
}

for date, group in momentum_panel.groupby("date"):
    for signal_name, (residual_column, momentum_column) in (
        residual_pairs.items()
    ):
        valid = group[
            [residual_column, momentum_column]
        ].dropna()

        if len(valid) < 30:
            continue

        residual_correlation_rows.append(
            {
                "date": date,
                "signal": signal_name,
                "correlation": valid[
                    residual_column
                ].corr(
                    valid[momentum_column],
                    method="spearman",
                ),
            }
        )

residual_correlation_summary = (
    pd.DataFrame(residual_correlation_rows)
    .groupby("signal")["correlation"]
    .agg(["count", "mean", "median", "std", "min", "max"])
)

residual_correlation_summary

,count,mean,median,std,min,max
signal,,,,,,
Raw residual,2639,0.080951,0.078152,0.047516,-0.118566,0.238967
Sector-neutral residual,2639,0.096981,0.089521,0.055893,-0.052967,0.322830


In [10]:
# Test the residual IC

incremental_signals = {
    "Idio-vol adjustment — incremental": (
        "risk_adjusted_mom_idio_incremental"
    ),
    "Idio-vol adjustment — sector-neutral incremental": (
        "risk_adjusted_mom_idio_sector_incremental"
    ),
}

incremental_ic_results = []

for signal_name, factor_column in incremental_signals.items():
    result = calculate_ic_by_horizon(
        panel=momentum_panel,
        factor_column=factor_column,
        forward_return_columns=[
            "forward_ret_1d",
            "forward_ret_5d",
        ],
        method="spearman",
        min_observations=30,
    )

    result["signal"] = signal_name
    incremental_ic_results.append(result)

incremental_ic_summary = pd.concat(
    incremental_ic_results,
    ignore_index=True,
)

incremental_ic_summary[
    [
        "signal",
        "forward_return_column",
        "count",
        "mean_ic",
        "std_ic",
        "ic_ir",
        "t_stat",
        "positive_fraction",
    ]
]

,signal,forward_return_column,count,mean_ic,std_ic,ic_ir,t_stat,positive_fraction
0,Idio-vol adjustment — incremental,forward_ret_1d,2638.0,0.003334,0.138048,0.024153,1.240549,0.520849
1,Idio-vol adjustment — incremental,forward_ret_5d,2634.0,-0.001343,0.135747,-0.009890,-0.507591,0.490888
2,Idio-vol adjustment — sector-neutral incremental,forward_ret_1d,2638.0,0.007318,0.125161,0.058469,3.003048,0.532221
3,Idio-vol adjustment — sector-neutral incremental,forward_ret_5d,2634.0,0.003428,0.126565,0.027087,1.390163,0.501898


### Diagnostic on the idio vol terciles

In [11]:
def assign_daily_terciles(series):
    result = pd.Series(
        pd.NA,
        index=series.index,
        dtype="Int64",
    )

    valid = series.dropna()

    if len(valid) < 30:
        return result

    ranks = valid.rank(
        method="first",
        pct=True,
    )

    result.loc[valid.index] = pd.cut(
        ranks,
        bins=[0.0, 1 / 3, 2 / 3, 1.0],
        labels=[1, 2, 3],
        include_lowest=True,
    ).astype("Int64")

    return result


momentum_panel["idio_vol_tercile"] = (
    momentum_panel
    .groupby("date")["idio_vol_63_raw"]
    .transform(assign_daily_terciles)
)

In [12]:
idio_vol_tercile_counts = (
    momentum_panel
    .dropna(subset=["idio_vol_tercile"])
    .groupby(["date", "idio_vol_tercile"], observed=True)
    .size()
    .groupby("idio_vol_tercile")
    .agg(["count", "mean", "min", "max"])
)

idio_vol_tercile_counts

,count,mean,min,max
idio_vol_tercile,,,,
1,2828,32.488685,32,33
2,2828,32.612801,32,33
3,2828,33.178571,33,34


In [13]:
# Calculate momentum IC within each tercile

momentum_by_risk_results = []

tercile_labels = {
    1: "Low idiosyncratic volatility",
    2: "Medium idiosyncratic volatility",
    3: "High idiosyncratic volatility",
}

for tercile, tercile_name in tercile_labels.items():
    tercile_panel = momentum_panel.loc[
        momentum_panel["idio_vol_tercile"] == tercile
    ].copy()

    result = calculate_ic_by_horizon(
        panel=tercile_panel,
        factor_column="mom_12_1m_z",
        forward_return_columns=[
            "forward_ret_1d",
            "forward_ret_5d",
        ],
        method="spearman",
        min_observations=20,
    )

    result["idio_vol_tercile"] = tercile_name
    momentum_by_risk_results.append(result)

momentum_by_risk_summary = pd.concat(
    momentum_by_risk_results,
    ignore_index=True,
)

momentum_by_risk_summary[
    [
        "idio_vol_tercile",
        "forward_return_column",
        "count",
        "mean_ic",
        "std_ic",
        "ic_ir",
        "t_stat",
        "positive_fraction",
    ]
]

,idio_vol_tercile,forward_return_column,count,mean_ic,std_ic,ic_ir,t_stat,positive_fraction
0,Low idiosyncratic volatility,forward_ret_1d,2638.0,0.019843,0.301139,0.065892,3.384332,0.531463
1,Low idiosyncratic volatility,forward_ret_5d,2634.0,0.015461,0.294953,0.052419,2.690289,0.533409
2,Medium idiosyncratic volatility,forward_ret_1d,2638.0,0.017934,0.297509,0.060280,3.096083,0.533359
3,Medium idiosyncratic volatility,forward_ret_5d,2634.0,0.022536,0.290535,0.077569,3.981017,0.538345
4,High idiosyncratic volatility,forward_ret_1d,2638.0,0.020393,0.331680,0.061484,3.157913,0.539803
5,High idiosyncratic volatility,forward_ret_5d,2634.0,0.024019,0.323539,0.074239,3.810116,0.542521


## Conclusion

This experiment tested whether scaling 12–1 momentum by total or idiosyncratic volatility improves its predictive power.

### Main findings

- All risk-adjusted variants produced ICs comparable to, but generally below, ordinary 12–1 momentum.
- The idiosyncratic-volatility-adjusted signal had an average daily rank correlation of **0.955** with ordinary momentum, indicating that
  it preserves almost the same cross-sectional ranking.
- Residualising the adjusted signal against momentum produced no robust incremental information:
  - raw incremental IC was **0.33%** at one day and **−0.13%** at five days;
  - sector-neutral incremental IC was **0.73%** at one day but only **0.34%** at five days.
- Momentum was not stronger among low-idiosyncratic-volatility stocks. At the five-day horizon, mean IC increased from **1.55%** in the low-risk tercile to **2.40%** in the high-risk tercile.

### Research decision

Idiosyncratic-volatility scaling introduces a modest low-risk tilt but does not improve momentum's predictive ranking or identify a stronger low-risk momentum regime.

Therefore, risk-adjusted momentum is not advanced as a standalone production factor, and no cost-aware backtest is required. Ordinary 12–1 momentum remains the preferred specification.

Volatility-based position sizing may still be investigated later as a portfolio-construction technique, but that is distinct from treating volatility-adjusted momentum as a new alpha signal.